# S.T.I.T.C.H — Floorplan Segmentation Training
Run each cell top to bottom.

In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2 — Check GPU
!nvidia-smi

In [ ]:
# CELL 3 — Unzip dataset from Drive
# Make sure dataset.zip is in the root of your Google Drive
import os

ZIP_PATH = '/content/drive/MyDrive/dataset.zip'   # change if yours is in a subfolder
EXTRACT_PATH = '/content/'

print('Unzipping dataset...')
!unzip -q {ZIP_PATH} -d {EXTRACT_PATH}
print('Done!')
print('Total images:', len(os.listdir('/content/dataset/images')))
print('Total masks: ', len(os.listdir('/content/dataset/masks')))

In [ ]:
# CELL 4 — Install dependencies
!pip install tqdm opencv-python-headless -q

In [ ]:
# CELL 5 — Model definition
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DoubleConv(3, 64)
        self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128)
        self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256)
        self.p3 = nn.MaxPool2d(2)
        self.b  = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b  = self.b(self.p3(d3))
        u3 = self.c3(torch.cat([self.u3(b), d3], dim=1))
        u2 = self.c2(torch.cat([self.u2(u3), d2], dim=1))
        u1 = self.c1(torch.cat([self.u1(u2), d1], dim=1))
        return self.out(u1)

print('✅ Model defined')

In [ ]:
# CELL 6 — Dataset
import os
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class FloorplanDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_names = sorted(os.listdir(img_dir))
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        name = self.img_names[idx]

        img = cv2.imread(os.path.join(self.img_dir, name))
        if img is None:
            return self.__getitem__(0)
        img = cv2.resize(img, (256, 256))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1)

        mask = cv2.imread(os.path.join(self.mask_dir, name), cv2.IMREAD_UNCHANGED)
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return img, mask

print('✅ Dataset defined')

In [ ]:
# CELL 7 — Training
import torch.optim as optim
from tqdm import tqdm

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

dataset = FloorplanDataset('/content/dataset/images', '/content/dataset/masks')
loader  = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)
print(f'Total images: {len(dataset)}')
print(f'Total batches: {len(loader)}')

model     = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn   = nn.BCEWithLogitsLoss()
epochs    = 15

for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}')

    for imgs, masks in loop:
        imgs  = imgs.to(device)
        masks = masks.to(device)
        preds = model(imgs)
        loss  = loss_fn(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(loader)
    print(f'Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}')

    # save checkpoint every epoch
    torch.save(model.state_dict(), '/content/unet.pth')

print('\n✅ TRAINING COMPLETE')

In [ ]:
# CELL 8 — Save model to Google Drive (so you don't lose it)
import shutil
shutil.copy('/content/unet.pth', '/content/drive/MyDrive/unet.pth')
print('✅ Model saved to Google Drive!')